# Implementing LSH on the N-most cited dataset

In [1]:
import json
from typing import Dict, Any
import numpy as np

In [2]:
from lsh import preprocess_lsh, lsh 
from lsh_utils.signatures import signatures
from lsh_utils.Jaccard_similarity import Jaccard_similarity_shingles

## Preprocess 

Keep ony the important part of the data

In [3]:
json_path_Nmost = '../data/processed/filtered_articles_Nmostcited.json'
data_Nmost = preprocess_lsh(dataset_path = json_path_Nmost)
print("preprocessing done")

Data succesfully loaded
preprocessing done


Save the preprocessed data into a json file

In [4]:
output_json_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Nmost, f, indent=4)

## Compute the signatures

Load the simplified data previously saved

In [3]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

Compute the signature matrix

In [21]:
q = 7 
b = 10
r = 10

signature_matrix_Nmost, idx_to_id_Nmost = signatures(
    doc_list=data_Nmost,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures: 100%|██████████| 24290/24290 [57:26<00:00,  7.05it/s]   

min hashing of the documents complete


Save the results

In [ ]:
np.save(file = f"../data/processed/signatures_lsh/Cat_q{q}_b{b}_r{r}", arr = signature_matrix_Nmost)
output_json_path = f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Nmost, f, indent=4)

## Perform the research of the most relevant documents using LSH

### Try with the parmeters : q = 7, b = 4, r = 5

load data and the saved signature matrix

In [5]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

q = 7 
b = 4
r = 5

try :
    signature_matrix_Nmost = np.load(f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}.npy")
    with open(f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id_Nmost: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [4]:
print(signature_matrix_Nmost.shape)

(20, 24290)


In [5]:
print(idx_to_id_Nmost[2])

{'id': '0911.0802', 'abstract': 'we construct analytic extensions of the pomeranskysenkov metrics with\nmultiple killing horizons and asymptotic regions we show that in our\nextensions the singularities associated to an obstruction to differentiability\nof the metric lie beyond event horizons we analyze the topology of the\nnonempty singular set which turns out to be parameterdependent we present\nnumerical evidence for stable causality of the domain of outer communications\nthe resulting global structure is somewhat reminiscent of that of kerr\nspacetime'}


In [6]:
print(type(data_Nmost))

<class 'list'>


Perform LSH to obtain the most similar documents to input (try with abstract of the first document as input)

In [6]:
Most_similar_20, Scores = lsh(
        input = data_Nmost[0]['clean_text'],
        article_list=data_Nmost,
        signature_matrix = signature_matrix_Nmost,
        idx_to_id = idx_to_id_Nmost,
        q = q,
        m = signature_matrix_Nmost.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
        )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures:   0%|          | 0/1 [00:00<?, ?it/s]

Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 43.69it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands:  25%|██▌       | 1/4 [00:00<00:00,  3.05it/s]

...

LSH Bands:  75%|███████▌  | 3/4 [00:00<00:00,  3.91it/s]

2.0000411692054345 % of signatures computed ... 



LSH Bands: 100%|██████████| 4/4 [00:01<00:00,  3.84it/s]


3.0000411692054345 % of signatures computed ... 

LSH successfully performed to find similar candidates
Calculation of the actual similarities ...


Calculating Similarities: 100%|██████████| 45/45 [00:00<00:00, 334.07it/s]


In [7]:
print("Most similar documents : " , Most_similar_20, '\n')
print("Scores : " , Scores)

Most similar documents :  ['1002.3982', '0909.1739', '1110.4738', '1202.1316', '0906.1523', '0904.3198', '0802.4221', '0808.3413', '1001.3651', '0903.3108', '0811.4571', '0704.3084', '0706.0322', '1003.3155', '0805.0332', '1111.5608', '0904.0380', '0802.1718', '0903.2733', '0807.0646', '0906.4370', '0810.3296', '1103.0885', '0901.4348', '1012.1201', '1010.5141', '0712.2716', '1108.2291', '1006.0106', '0808.1161', '1203.1672', '0907.5115', '0809.5268', '0803.1447', '1110.6838', '1005.4655', '1011.5154', '0806.3825', '1005.3266', '0903.4375', '1003.3590', '0806.4688', '0807.4146', '1005.1132', '0902.1539'] 

Scores :  [1.0, 0.020854021847070508, 0.016877637130801686, 0.016353229762878167, 0.013874066168623266, 0.013071895424836602, 0.012057877813504822, 0.011959521619135235, 0.011318242343541944, 0.010550996483001172, 0.009925558312655087, 0.009812667261373774, 0.008169934640522876, 0.007789678675754625, 0.007759456838021339, 0.006622516556291391, 0.0061403508771929825, 0.006074411541381

In [11]:
data_Nmost[0]['id'] # id of the first document (input)

'1002.3982'

We notice that the document has a 100% similarity with himself ! Let's have a look to the second most relevant document found

In [8]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")

In [10]:
Id_list = [doc["id"] for doc in idx_to_id_Nmost]
i = find_index(Id_list,Most_similar_20[1])
abstract_20 = data_Nmost[i]["clean_text"]
print(abstract_20)

parityodd domains corresponding to nontrivial topological solutions of the
qcd vacuum might be created during relativistic heavy ion collisions these
domains are predicted to lead to charge separation of quarks along the systems
orbital momentum axis we investigate a three particle azimuthal correlator
which is a p even observable but directly sensitive to the charge separation
effect we report measurements of charged hadrons near centerofmass rapidity
with this observable in auau and cucu collisions at sqrtsnn200 gev
using the star detector a signal consistent with several expectations from the
theory is detected we discuss possible contributions from other effects that
are not related to parity violation


This document is about neutrinos

In [11]:
print(data_Nmost[0]["clean_text"])

new particles at the tev scale can decay hadronically with strongly
collimated jets thus the standard reconstruction methods based on
invariantmasses of wellseparated jets can fail we discuss how to identify
such particles in pp collisions at the lhc using jet shapes which help to
reduce the contribution of qcdinduced events we focus on a rather generic
example x to ttbar to hadrons with x being a heavy particle but the approach
is well suited for reconstruction of other decay channels characterized by a
cascade decay of known states


The original document talks about particles : which is a similar subject !!

### Try with the set of parameters : q = 7, b = 10, r = 10

In [3]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

q = 7 
b = 10
r = 10

try :
    signature_matrix_Nmost = np.load(f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}.npy")
    with open(f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id_Nmost: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [16]:
print(signature_matrix_Nmost.shape)

(100, 24290)


Perform LSH to obtain the most similar documents to input (try with abstract of the first document as input)

In [4]:
Most_similar_100, Scores = lsh(
        input = data_Nmost[0]['clean_text'],
        article_list=data_Nmost,
        signature_matrix = signature_matrix_Nmost,
        idx_to_id = idx_to_id_Nmost,
        q=q,
        m = signature_matrix_Nmost.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
        )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures:   0%|          | 0/1 [00:00<?, ?it/s]

Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 10.56it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands:  10%|█         | 1/10 [00:00<00:03,  2.87it/s]

...

LSH Bands:  90%|█████████ | 9/10 [00:03<00:00,  2.47it/s]

8.000041169205435 % of signatures computed ... 



LSH Bands: 100%|██████████| 10/10 [00:04<00:00,  2.50it/s]


9.000041169205435 % of signatures computed ... 

LSH successfully performed to find similar candidates
Calculation of the actual similarities ...


Calculating Similarities: 100%|██████████| 107/107 [00:00<00:00, 304.72it/s]


In [15]:
print("Most similar documents : " , Most_similar_100, '\n')
print("Scores : " , Scores)

Most similar documents :  ['1002.3982', '1001.4577', '1207.4235', '0909.4397', '0901.4533', '1012.4840', '0807.3834', '1008.1632', '0712.4328', '1107.1244', '0711.1365', '0709.0007', '0711.4754', '1001.3651', '1112.3024', '1102.2777', '1110.6249', '0712.1028', '0708.4003', '1102.0342', '1110.4372', '0907.1269', '1002.2424', '1107.1997', '0909.2937', '0908.0880', '0907.1014', '0804.1585', '0911.1535', '1207.1468', '0810.4846', '0712.1026', '0806.4175', '0709.0731', '1201.3339', '1008.5243', '0904.1554', '1010.0055', '0804.1648', '0807.1939', '0911.5281', '0803.4323', '1202.4057', '1106.5493', '0705.0572', '0912.0399', '0805.4451', '0905.1890', '0710.2435', '1004.0861', '0705.2629', '1112.3351', '0909.1063', '1003.4660', '0909.4766', '0801.0778', '0806.0050', '0910.2076', '0706.4153', '1004.0162', '0805.2466', '0910.0554', '0710.0915', '0901.3946', '1005.1909', '1103.1740', '0906.2165', '0710.2498', '0911.0352', '0903.1286', '0807.1108', '1004.2045', '0907.5008', '0907.2997', '1005.0398'

In [16]:
data_Nmost[0]['id'] # id of the first document (input)

'1002.3982'

In [18]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")
Id_list = [doc["id"] for doc in idx_to_id_Nmost]
i = find_index(Id_list,Most_similar_100[1])
abstract_100 = data_Nmost[i]["clean_text"]
print(abstract_100)

we report a search for single top quark production with the cdf ii detector
using 21 fb1 of integrated luminosity of pbar p collisions at sqrts196
tev the data selected consist of events characterized by large energy
imbalance in the transverse plane and hadronic jets and no identified
electrons and muons so the sample is enriched in w  tau nu decays in order
to suppress backgrounds additional kinematic and topological requirements are
imposed through a neural network and at least one of the jets must be
identified as a bquark jet we measure an excess of signallike events in
agreement with the standard model prediction but inconsistent with a model
without single top quark production by 21 standard deviations sigma with a
median expected sensitivity of 14 sigma assuming a top quark mass of 175
gevc2 and ascribing the excess to single top quark production the cross
section is measured to be 492522statsystpb consistent with
measurements performed in independent datasets and with the stan

In [20]:
print(data_Nmost[0]["clean_text"])

new particles at the tev scale can decay hadronically with strongly
collimated jets thus the standard reconstruction methods based on
invariantmasses of wellseparated jets can fail we discuss how to identify
such particles in pp collisions at the lhc using jet shapes which help to
reduce the contribution of qcdinduced events we focus on a rather generic
example x to ttbar to hadrons with x being a heavy particle but the approach
is well suited for reconstruction of other decay channels characterized by a
cascade decay of known states


We also find a similar subject for the most relevant document

### Comparing the results of b,r = 4,5 and b,r = 10,10 for the second most similar document

In [21]:
from lsh_utils.shingle import shingle

In [23]:
input_content = data_Nmost[0]["clean_text"]
shingle_input = shingle(q=7, text = input_content)
shingle_20 = shingle(q=7, text = abstract_20)
shingle_100 = shingle(q=7,text = abstract_100)

In [24]:
Sim_20 = Jaccard_similarity_shingles(shingles_list_A=shingle_input,shingles_list_B=shingle_20)
Sim_100 = Jaccard_similarity_shingles(shingles_list_A=shingle_input, shingles_list_B=shingle_100)

In [25]:
print(Sim_20,Sim_100)

0.020854021847070508 0.021702838063439065


The set of parameters with b,r = 10,10 has better results !

### Try lsh_n

In [5]:
from lsh import lsh_n

In [6]:
n=10
Most_similar_100_n = lsh_n(
    n,
    input = data_Nmost[0]['clean_text'],
        article_list=data_Nmost,
        signature_matrix = signature_matrix_Nmost,
        idx_to_id = idx_to_id_Nmost,
        q=q,
        m = signature_matrix_Nmost.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
)

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 12.23it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands:  10%|█         | 1/10 [00:00<00:02,  4.19it/s]

4.1169205434335116e-05 % of signatures computed ... 



...

LSH Bands:  90%|█████████ | 9/10 [00:02<00:00,  3.33it/s]

8.000041169205435 % of signatures computed ... 



LSH Bands: 100%|██████████| 10/10 [00:02<00:00,  3.53it/s]


9.000041169205435 % of signatures computed ... 

LSH successfully performed to find similar candidates
Calculation of the actual similarities ...


Calculating Similarities: 100%|██████████| 107/107 [00:00<00:00, 408.27it/s]


In [7]:
Most_similar_100_n

['1002.3982',
 '1001.4577',
 '1207.4235',
 '0909.4397',
 '0901.4533',
 '1012.4840',
 '0807.3834',
 '1008.1632',
 '0712.4328',
 '1107.1244']

In [8]:
Most_similar_100

['1002.3982',
 '1001.4577',
 '1207.4235',
 '0909.4397',
 '0901.4533',
 '1012.4840',
 '0807.3834',
 '1008.1632',
 '0712.4328',
 '1107.1244',
 '0711.1365',
 '0709.0007',
 '0711.4754',
 '1001.3651',
 '1112.3024',
 '1102.2777',
 '1110.6249',
 '0712.1028',
 '0708.4003',
 '1102.0342',
 '1110.4372',
 '0907.1269',
 '1002.2424',
 '1107.1997',
 '0909.2937',
 '0908.0880',
 '0907.1014',
 '0804.1585',
 '0911.1535',
 '1207.1468',
 '0810.4846',
 '0712.1026',
 '0806.4175',
 '0709.0731',
 '1201.3339',
 '1008.5243',
 '0904.1554',
 '1010.0055',
 '0804.1648',
 '0807.1939',
 '0911.5281',
 '0803.4323',
 '1202.4057',
 '1106.5493',
 '0705.0572',
 '0912.0399',
 '0805.4451',
 '0905.1890',
 '0710.2435',
 '1004.0861',
 '0705.2629',
 '1112.3351',
 '0909.1063',
 '1003.4660',
 '0909.4766',
 '0801.0778',
 '0806.0050',
 '0910.2076',
 '0706.4153',
 '1004.0162',
 '0805.2466',
 '0910.0554',
 '0710.0915',
 '0901.3946',
 '1005.1909',
 '1103.1740',
 '0906.2165',
 '0710.2498',
 '0911.0352',
 '0903.1286',
 '0807.1108',
 '1004

# Compute signatures for K most cited per category dataset

In [30]:
json_path_Cat = '../data/processed/filtered_articles_Cat.json'
data_Cat = preprocess_lsh(dataset_path = json_path_Cat)
print("preprocessing done")

Data succesfully loaded
preprocessing done


In [ ]:
output_json_path = '../data/processed/sub_datasets_lsh/data_Cat.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Cat, f, indent=4)

Compute signatures

In [33]:
q = 7 
b = 10
r = 10

signature_matrix_Cat, idx_to_id_Cat = signatures(
    doc_list=data_Cat,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures: 100%|██████████| 24290/24290 [48:09<00:00,  8.41it/s] 

min hashing of the documents complete


Save the signature matrix and the idx_to_id dictionnary

In [34]:
np.save(file = f"../data/processed/signatures_lsh/Cat_q{q}_b{b}_r{r}", arr = signature_matrix_Cat)
output_json_path = f"../data/processed/signatures_lsh/idx_to_id_Cat_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Cat, f, indent=4)

# Compute signatures for Quartiles dataset

In [35]:
json_path_Quartiles = '../data/processed/filtered_articles_Quartiles.json'
data_Quartiles = preprocess_lsh(dataset_path = json_path_Quartiles)
print("preprocessing done")

Data succesfully loaded
preprocessing done


In [36]:
output_json_path = '../data/processed/sub_datasets_lsh/data_Quartiles.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Quartiles, f, indent=4)

Compute signatures

In [37]:
q = 7 
b = 10
r = 10

signature_matrix_Quartiles, idx_to_id_Quartiles = signatures(
    doc_list=data_Quartiles,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures:  36%|███▌      | 15395/42834 [55:48<1:01:49,  7.40it/s]   

: 

Save the signature matrix and the idx_to_id dictionnary

In [ ]:
np.save(file = f"../data/processed/signatures_lsh/stratified_q{q}_b{b}_r{r}", arr = signature_matrix_Quartiles)
output_json_path = f"../data/processed/signatures_lsh/idx_to_id_stratified_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Quartiles, f, indent=4)

# Compute signatures for Stratified dataset

In [ ]:
json_path_stratified = '../data/processed/filtered_articles_stratified.json'
data_stratified = preprocess_lsh(dataset_path = json_path_stratified)
print("preprocessing done")

In [ ]:
output_json_path = '../data/processed/sub_datasets_lsh/data_stratified.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_stratified, f, indent=4)

Compute signatures

In [ ]:
q = 7 
b = 10
r = 10

signature_matrix_stratified, idx_to_id_stratified = signatures(
    doc_list=data_stratified,
    shingle_size = q,
    signature_size = b*r
    )

Save the signature matrix and the idx_to_id dictionnary

In [ ]:
np.save(file = f"../data/processed/signatures_lsh/stratified_q{q}_b{b}_r{r}", arr = signature_matrix_stratified)
output_json_path = f"../data/processed/signatures_lsh/idx_to_id_stratified_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_stratified, f, indent=4)